# 🎯 Phase 1: Milestone Exam Solutions

> **Algorithmic Thinking & Python Foundations**
>
> This notebook contains comprehensive solutions for all four Phase Milestone Exam questions.
> Each solution demonstrates the synthesis of concepts from Days 1-12.

---

## Question 1: The E-Commerce Cart

**Combines**: Lists (Day 5), Dictionaries (Day 8), Functions (Day 11), Conditionals (Day 9)

**Scenario**: Build a shopping cart system with:
1. Products stored as dictionaries with `name`, `price`, and `stock`
2. Cart as a list of tuples: `(product_name, quantity)`
3. Functions for adding items, calculating totals, and applying discounts

In [1]:
# Sample product data
products = {
    "laptop": {"name": "Laptop Pro", "price": 999.99, "stock": 10},
    "mouse": {"name": "Wireless Mouse", "price": 29.99, "stock": 50},
    "keyboard": {"name": "Mechanical Keyboard", "price": 79.99, "stock": 25},
}

In [2]:
def add_to_cart(cart, product_key, quantity, products_catalog):
    """
    Add an item to the shopping cart if it exists and is in stock.

    Args:
        cart: List of (product_key, quantity) tuples
        product_key: Key to look up in products catalog
        quantity: Number of items to add
        products_catalog: Dictionary of available products

    Returns:
        tuple: (success: bool, message: str)
    """
    # Check if product exists
    if product_key not in products_catalog:
        return False, f"Product '{product_key}' not found in catalog"

    product = products_catalog[product_key]

    # Check if quantity is valid
    if quantity <= 0:
        return False, "Quantity must be positive"

    # Check stock availability
    if quantity > product["stock"]:
        return False, f"Insufficient stock. Only {product['stock']} available"

    # Add to cart (check if item already exists)
    for i, (key, qty) in enumerate(cart):
        if key == product_key:
            new_qty = qty + quantity
            if new_qty > product["stock"]:
                return (
                    False,
                    f"Cannot add {quantity} more. Only {product['stock'] - qty} available",
                )
            cart[i] = (product_key, new_qty)
            return True, f"Updated {product['name']} quantity to {new_qty}"

    cart.append((product_key, quantity))
    return True, f"Added {quantity}x {product['name']} to cart"

In [3]:
def calculate_total(cart, products_catalog):
    """
    Calculate the subtotal of all items in the cart.

    Args:
        cart: List of (product_key, quantity) tuples
        products_catalog: Dictionary of available products

    Returns:
        float: Subtotal of all items
    """
    subtotal = 0.0

    for product_key, quantity in cart:
        if product_key in products_catalog:
            price = products_catalog[product_key]["price"]
            subtotal += price * quantity

    return round(subtotal, 2)

In [4]:
def apply_discount(total, membership_tier):
    """
    Apply membership discount to the total.

    Args:
        total: Cart subtotal
        membership_tier: 'bronze', 'silver', or 'gold'

    Returns:
        float: Discounted total
    """
    discount_rates = {
        "bronze": 0.05,  # 5% off
        "silver": 0.10,  # 10% off
        "gold": 0.15,  # 15% off
    }

    # Default to 0% if tier not recognized
    discount = discount_rates.get(membership_tier.lower(), 0)

    discounted_total = total * (1 - discount)
    return round(discounted_total, 2)

In [5]:
# Test the E-Commerce Cart System
print("=" * 50)
print("E-COMMERCE CART SYSTEM TEST")
print("=" * 50)

cart = []

# Add items
success, msg = add_to_cart(cart, "laptop", 2, products)
print(f"Add laptop x2: {msg}")

success, msg = add_to_cart(cart, "mouse", 3, products)
print(f"Add mouse x3: {msg}")

# Try to add invalid product
success, msg = add_to_cart(cart, "tablet", 1, products)
print(f"Add tablet x1: {msg}")

# Try to add more than stock
success, msg = add_to_cart(cart, "keyboard", 30, products)
print(f"Add keyboard x30: {msg}")

print(f"\nCart contents: {cart}")

# Calculate totals
subtotal = calculate_total(cart, products)
print(f"\nSubtotal: ${subtotal:,.2f}")

# Apply discounts
for tier in ["bronze", "silver", "gold"]:
    final = apply_discount(subtotal, tier)
    print(f"{tier.capitalize()} member total: ${final:,.2f}")

# Verify expected calculation: (999.99*2 + 29.99*3) * 0.90 for silver
expected = (999.99 * 2 + 29.99 * 3) * 0.90
print(f"\nExpected silver total: ${expected:,.2f}")

E-COMMERCE CART SYSTEM TEST
Add laptop x2: Added 2x Laptop Pro to cart
Add mouse x3: Added 3x Wireless Mouse to cart
Add tablet x1: Product 'tablet' not found in catalog
Add keyboard x30: Insufficient stock. Only 25 available

Cart contents: [('laptop', 2), ('mouse', 3)]

Subtotal: $2,089.95
Bronze member total: $1,985.45
Silver member total: $1,880.95
Gold member total: $1,776.46

Expected silver total: $1,880.95


---

## Question 2: The Data Cleaning Pipeline

**Combines**: Strings (Day 4), Sets (Day 7), Functions (Day 11), List Comprehensions (Day 12)

**Scenario**: Clean messy customer data with name normalization, email cleaning, phone validation, and duplicate detection.

In [6]:
# Sample messy customer data
raw_customers = [
    "  JOHN DOE, john.doe@email.com, 555-123-4567  ",
    "jane smith, JANE@COMPANY.ORG, 555.987.6543",
    "Bob Wilson, bob@test.com, 555-123-4567",  # Duplicate phone
    "  alice jones, alice@email.com, invalid-phone",
    "john doe, johndoe@email.com, 555-111-2222",  # Duplicate name
]

In [7]:
def clean_name(name):
    """
    Clean and normalize a name string.

    Args:
        name: Raw name string

    Returns:
        str: Title-cased, trimmed name
    """
    return name.strip().title()

In [8]:
def clean_email(email):
    """
    Clean and normalize an email address.

    Args:
        email: Raw email string

    Returns:
        str: Lowercase, trimmed email
    """
    return email.strip().lower()

In [9]:
def clean_phone(phone):
    """
    Clean and validate a phone number.

    Args:
        phone: Raw phone string

    Returns:
        str or None: Cleaned 10-digit phone number, or None if invalid
    """
    # Extract only digits using list comprehension
    digits = "".join([c for c in phone if c.isdigit()])

    # Valid phone numbers have exactly 10 digits
    if len(digits) == 10:
        return digits
    return None

In [10]:
def parse_customer(raw_string):
    """
    Parse a raw customer string into a clean dictionary.

    Args:
        raw_string: Comma-separated string with name, email, phone

    Returns:
        dict: Customer data with 'name', 'email', 'phone' keys
    """
    parts = raw_string.split(",")

    # Handle malformed data
    if len(parts) != 3:
        return {"name": None, "email": None, "phone": None}

    name_raw, email_raw, phone_raw = parts

    return {
        "name": clean_name(name_raw),
        "email": clean_email(email_raw),
        "phone": clean_phone(phone_raw),
    }

In [11]:
def find_duplicates(customers):
    """
    Find duplicate names in the customer list.

    Args:
        customers: List of customer dictionaries

    Returns:
        set: Set of names that appear more than once
    """
    seen = set()
    duplicates = set()

    for customer in customers:
        name = customer["name"]
        if name in seen:
            duplicates.add(name)
        seen.add(name)

    return duplicates

In [12]:
# Test the Data Cleaning Pipeline
print("=" * 50)
print("DATA CLEANING PIPELINE TEST")
print("=" * 50)

# Parse all customers using list comprehension
customers = [parse_customer(r) for r in raw_customers]

print("\nAll parsed customers:")
for i, c in enumerate(customers, 1):
    print(f"  {i}. {c}")

# Filter valid customers (those with valid phone numbers)
valid_customers = [c for c in customers if c["phone"]]

print(f"\nValid customers (with phone): {len(valid_customers)}")
for c in valid_customers:
    print(f"  - {c['name']}: {c['email']}, {c['phone']}")

# Get unique names using set comprehension
unique_names = {c["name"] for c in valid_customers}
print(f"\nUnique names: {unique_names}")

# Find duplicate names
duplicates = find_duplicates(customers)
print(f"\nDuplicate names: {duplicates}")

DATA CLEANING PIPELINE TEST

All parsed customers:
  1. {'name': 'John Doe', 'email': 'john.doe@email.com', 'phone': '5551234567'}
  2. {'name': 'Jane Smith', 'email': 'jane@company.org', 'phone': '5559876543'}
  3. {'name': 'Bob Wilson', 'email': 'bob@test.com', 'phone': '5551234567'}
  4. {'name': 'Alice Jones', 'email': 'alice@email.com', 'phone': None}
  5. {'name': 'John Doe', 'email': 'johndoe@email.com', 'phone': '5551112222'}

Valid customers (with phone): 4
  - John Doe: john.doe@email.com, 5551234567
  - Jane Smith: jane@company.org, 5559876543
  - Bob Wilson: bob@test.com, 5551234567
  - John Doe: johndoe@email.com, 5551112222

Unique names: {'Bob Wilson', 'Jane Smith', 'John Doe'}

Duplicate names: {'John Doe'}


---

## Question 3: The Sales Analytics Dashboard

**Combines**: Loops (Day 10), Dictionaries (Day 8), Conditionals (Day 9), Tuples (Day 6)

**Scenario**: Build analytics functions for monthly sales data including aggregation by region, top category identification, and flexible filtering.

In [13]:
from collections import defaultdict

# Sample sales data: (date, region, category, amount)
sales_log = [
    ("2024-01-15", "North", "Electronics", 1200),
    ("2024-01-18", "South", "Clothing", 450),
    ("2024-01-22", "North", "Electronics", 890),
    ("2024-02-10", "West", "Electronics", 2100),
    ("2024-02-15", "North", "Clothing", 780),
    ("2024-02-20", "South", "Electronics", 1350),
    ("2024-03-05", "East", "Electronics", 1800),
    ("2024-03-12", "North", "Furniture", 2500),
    ("2024-03-18", "West", "Clothing", 620),
    ("2024-03-25", "South", "Furniture", 1950),
]

In [14]:
def total_by_region(sales):
    """
    Calculate total sales by region.

    Args:
        sales: List of (date, region, category, amount) tuples

    Returns:
        dict: Mapping of region name to total sales
    """
    region_totals = defaultdict(float)

    for date, region, category, amount in sales:
        region_totals[region] += amount

    return dict(region_totals)

In [15]:
def top_category(sales):
    """
    Find the category with the highest total sales.

    Args:
        sales: List of (date, region, category, amount) tuples

    Returns:
        tuple: (category_name, total_amount)
    """
    category_totals = defaultdict(float)

    for date, region, category, amount in sales:
        category_totals[category] += amount

    if not category_totals:
        return (None, 0)

    # Find the max using max() with key function
    best_category = max(category_totals.keys(), key=lambda c: category_totals[c])
    return (best_category, category_totals[best_category])

In [16]:
def monthly_growth(sales):
    """
    Calculate month-over-month growth percentages.

    Args:
        sales: List of (date, region, category, amount) tuples

    Returns:
        dict: Mapping of month to growth percentage from previous month
    """
    # First, aggregate by month
    monthly_totals = defaultdict(float)

    for date, region, category, amount in sales:
        month = date[:7]  # Extract "YYYY-MM"
        monthly_totals[month] += amount

    # Sort months chronologically
    sorted_months = sorted(monthly_totals.keys())

    # Calculate growth
    growth = {}
    for i, month in enumerate(sorted_months):
        if i == 0:
            growth[month] = None  # No previous month to compare
        else:
            prev_month = sorted_months[i - 1]
            prev_total = monthly_totals[prev_month]
            curr_total = monthly_totals[month]
            if prev_total > 0:
                growth[month] = round(((curr_total - prev_total) / prev_total) * 100, 2)
            else:
                growth[month] = None

    return growth

In [17]:
def filter_sales(sales, min_amount=None, region=None, category=None):
    """
    Filter sales based on optional criteria.

    Args:
        sales: List of (date, region, category, amount) tuples
        min_amount: Minimum amount filter (optional)
        region: Region filter (optional)
        category: Category filter (optional)

    Returns:
        list: Filtered list of sales tuples
    """
    result = []

    for sale in sales:
        date, sale_region, sale_category, amount = sale

        # Apply all filters (None means no filter for that criterion)
        if min_amount is not None and amount < min_amount:
            continue
        if region is not None and sale_region != region:
            continue
        if category is not None and sale_category != category:
            continue

        result.append(sale)

    return result

In [18]:
# Test the Sales Analytics Dashboard
print("=" * 50)
print("SALES ANALYTICS DASHBOARD TEST")
print("=" * 50)

# Total by region
by_region = total_by_region(sales_log)
print("\nSales by Region:")
for region, total in sorted(by_region.items(), key=lambda x: -x[1]):
    print(f"  {region}: ${total:,.2f}")

# Top category
best = top_category(sales_log)
print(f"\nTop Category: {best[0]} (${best[1]:,.2f})")

# Monthly growth
growth = monthly_growth(sales_log)
print("\nMonthly Growth:")
for month, pct in growth.items():
    if pct is None:
        print(f"  {month}: (baseline)")
    else:
        arrow = "↑" if pct > 0 else "↓" if pct < 0 else "→"
        print(f"  {month}: {arrow} {pct:+.1f}%")

# Filtered views
print("\nFiltered: North region, $1000+ sales")
big_north = filter_sales(sales_log, min_amount=1000, region="North")
for sale in big_north:
    print(f"  {sale}")

print("\nFiltered: Electronics category only")
electronics = filter_sales(sales_log, category="Electronics")
for sale in electronics:
    print(f"  {sale}")

SALES ANALYTICS DASHBOARD TEST

Sales by Region:
  North: $5,370.00
  South: $3,750.00
  West: $2,720.00
  East: $1,800.00

Top Category: Electronics ($7,340.00)

Monthly Growth:
  2024-01: (baseline)
  2024-02: ↑ +66.5%
  2024-03: ↑ +62.4%

Filtered: North region, $1000+ sales
  ('2024-01-15', 'North', 'Electronics', 1200)
  ('2024-03-12', 'North', 'Furniture', 2500)

Filtered: Electronics category only
  ('2024-01-15', 'North', 'Electronics', 1200)
  ('2024-01-22', 'North', 'Electronics', 890)
  ('2024-02-10', 'West', 'Electronics', 2100)
  ('2024-02-20', 'South', 'Electronics', 1350)
  ('2024-03-05', 'East', 'Electronics', 1800)


---

## Question 4: The Password Policy Enforcer

**Combines**: Strings (Day 4), Functions (Day 11), Conditionals (Day 9), Operators (Day 3)

**Scenario**: Implement a configurable password validation system with strength scoring and pattern detection.

In [19]:
class PasswordPolicy:
    """
    Configurable password validation and strength scoring.

    Attributes:
        min_length: Minimum required password length
        require_upper: Require at least one uppercase letter
        require_lower: Require at least one lowercase letter
        require_digit: Require at least one digit
        require_special: Require at least one special character
        special_chars: Set of allowed special characters
    """

    # Common keyboard patterns to detect
    KEYBOARD_PATTERNS = [
        "qwerty",
        "qwertz",
        "azerty",
        "asdf",
        "zxcv",
        "1234",
        "2345",
        "3456",
        "4567",
        "5678",
        "6789",
        "7890",
        "0987",
        "9876",
        "8765",
        "7654",
        "6543",
        "5432",
        "4321",
    ]

    def __init__(
        self,
        min_length=8,
        require_upper=True,
        require_lower=True,
        require_digit=True,
        require_special=False,
        special_chars="!@#$%^&*",
    ):
        self.min_length = min_length
        self.require_upper = require_upper
        self.require_lower = require_lower
        self.require_digit = require_digit
        self.require_special = require_special
        self.special_chars = set(special_chars)

    def validate(self, password):
        """
        Validate password against the policy.

        Args:
            password: The password string to validate

        Returns:
            tuple: (is_valid: bool, errors: list of error messages)
        """
        errors = []

        # Length check
        if len(password) < self.min_length:
            errors.append(f"Password must be at least {self.min_length} characters")

        # Uppercase check
        if self.require_upper and not any(c.isupper() for c in password):
            errors.append("Password must contain an uppercase letter")

        # Lowercase check
        if self.require_lower and not any(c.islower() for c in password):
            errors.append("Password must contain a lowercase letter")

        # Digit check
        if self.require_digit and not any(c.isdigit() for c in password):
            errors.append("Password must contain a digit")

        # Special character check
        if self.require_special:
            if not any(c in self.special_chars for c in password):
                errors.append(
                    f"Password must contain a special character ({self.special_chars})"
                )

        return (len(errors) == 0, errors)

    def _has_sequential_chars(self, password, length=3):
        """Check for sequential characters like 'abc' or '123'."""
        for i in range(len(password) - length + 1):
            substring = password[i : i + length]
            # Check if all chars are consecutive (ascending)
            if all(
                ord(substring[j + 1]) == ord(substring[j]) + 1
                for j in range(len(substring) - 1)
            ):
                return True
            # Check descending
            if all(
                ord(substring[j + 1]) == ord(substring[j]) - 1
                for j in range(len(substring) - 1)
            ):
                return True
        return False

    def _has_repeated_chars(self, password, length=3):
        """Check for repeated characters like 'aaa' or '111'."""
        for i in range(len(password) - length + 1):
            substring = password[i : i + length]
            if len(set(substring)) == 1:  # All same character
                return True
        return False

    def _has_keyboard_pattern(self, password):
        """Check for common keyboard patterns."""
        lower_pwd = password.lower()
        for pattern in self.KEYBOARD_PATTERNS:
            if pattern in lower_pwd:
                return True
        return False

    def strength_score(self, password):
        """
        Calculate password strength score from 0-100.

        Scoring breakdown:
        - Length: up to 20 points (2 points per char, max at 10 chars)
        - Character variety: up to 40 points (10 each for upper, lower, digit, special)
        - No common patterns: up to 40 points (deduct for patterns found)

        Args:
            password: The password string to score

        Returns:
            int: Score from 0-100
        """
        score = 0

        # Length score (up to 20 points)
        length_score = min(len(password) * 2, 20)
        score += length_score

        # Character variety score (up to 40 points)
        variety_score = 0
        if any(c.isupper() for c in password):
            variety_score += 10
        if any(c.islower() for c in password):
            variety_score += 10
        if any(c.isdigit() for c in password):
            variety_score += 10
        if any(c in self.special_chars for c in password):
            variety_score += 10
        score += variety_score

        # Pattern detection (up to 40 points, deduct for bad patterns)
        pattern_score = 40
        if self._has_sequential_chars(password):
            pattern_score -= 15
        if self._has_repeated_chars(password):
            pattern_score -= 15
        if self._has_keyboard_pattern(password):
            pattern_score -= 20
        pattern_score = max(0, pattern_score)  # Don't go negative
        score += pattern_score

        return score

In [20]:
# Test the Password Policy Enforcer
print("=" * 50)
print("PASSWORD POLICY ENFORCER TEST")
print("=" * 50)

# Create a strict policy
policy = PasswordPolicy(min_length=10, require_special=True)

test_passwords = [
    "short",  # Too short, missing requirements
    "password123",  # No uppercase, no special
    "PASSWORD123",  # No lowercase, no special
    "Passw0rd!",  # Too short (9 chars)
    "SecureP@ss123",  # Valid!
    "MyP@ssw0rd!!",  # Valid!
    "Qwerty123!@",  # Valid but has keyboard pattern
    "Abc12345!!!",  # Valid but has sequential chars
    "Aaaa1234!@#",  # Valid but has repeated chars
]

print("\nValidation Results:")
print("-" * 50)

for pwd in test_passwords:
    is_valid, errors = policy.validate(pwd)
    score = policy.strength_score(pwd)
    status = "✅" if is_valid else "❌"

    print(f"{status} '{pwd}'")
    print(f"   Valid: {is_valid}, Score: {score}/100")
    if errors:
        for err in errors:
            print(f"   → {err}")
    print()

# Test with default (less strict) policy
print("\n" + "=" * 50)
print("DEFAULT POLICY (no special required)")
print("=" * 50)

default_policy = PasswordPolicy()  # min_length=8, no special required

for pwd in ["Password1", "MySecret99", "Test1234"]:
    is_valid, errors = default_policy.validate(pwd)
    score = default_policy.strength_score(pwd)
    print(f"'{pwd}': Valid={is_valid}, Score={score}/100")

PASSWORD POLICY ENFORCER TEST

Validation Results:
--------------------------------------------------
❌ 'short'
   Valid: False, Score: 60/100
   → Password must be at least 10 characters
   → Password must contain an uppercase letter
   → Password must contain a digit
   → Password must contain a special character ({'%', '*', '@', '!', '$', '^', '#', '&'})

❌ 'password123'
   Valid: False, Score: 65/100
   → Password must contain an uppercase letter
   → Password must contain a special character ({'%', '*', '@', '!', '$', '^', '#', '&'})

❌ 'PASSWORD123'
   Valid: False, Score: 65/100
   → Password must contain a lowercase letter
   → Password must contain a special character ({'%', '*', '@', '!', '$', '^', '#', '&'})

❌ 'Passw0rd!'
   Valid: False, Score: 98/100
   → Password must be at least 10 characters

✅ 'SecureP@ss123'
   Valid: True, Score: 85/100

✅ 'MyP@ssw0rd!!'
   Valid: True, Score: 100/100

✅ 'Qwerty123!@'
   Valid: True, Score: 65/100

✅ 'Abc12345!!!'
   Valid: True, Sc

---

## 🎓 Summary

This notebook demonstrated solutions to all four Phase 1 Milestone Exam questions:

1. **E-Commerce Cart**: Combined lists, dictionaries, functions, and conditionals
2. **Data Cleaning Pipeline**: Used strings, sets, and list comprehensions
3. **Sales Analytics Dashboard**: Applied loops, tuples, and aggregation patterns
4. **Password Policy Enforcer**: Built a class with validation and scoring logic

Each solution follows Python best practices including:
- Clear docstrings explaining purpose, args, and returns
- Edge case handling (empty inputs, invalid data)
- Modular, reusable function design
- Type-appropriate data structures (lists vs sets vs dicts)